# RLEF Alignment (TinyLlama-1.1B)
**Working Draft**

Aligning a local 1B model to produce optimized SQL using execution feedback logs. 
Comparing PPO (Reward Modeling) vs DPO.

In [ ]:
!pip install -q transformers peft trl bitsandbytes datasets

[notice] A new release of pip is available: 23.2.1 -> 24.0
[notice] To update, run: pip install --upgrade pip

## 1. Initial Model Load (VRAM Validation)
Loading base TinyLlama. Checking VRAM constraints before dropping to 4-bit.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# TODO: Test full precision
# model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

# ... Update: OOM encountered at batch size 4. 
# Reverting to QLoRA (NF4) with 8-bit paged optimizer.

In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Applying LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,293,760 || all params: 1,102,342,144 || trainable%: 0.2080806121

## 2. Preference Parsing
Parsing `episode_log.json` into preference pairs `(prompt, chosen, rejected)`.
Filtering instances where `optimized_cost` failed to outperform `baseline_cost` (reward <= 0).

In [ ]:
import json
from datasets import Dataset

pairs = {"prompt": [], "chosen": [], "rejected": []}

with open("../src/rlef/training_logs/episode_log.json", "r") as f:
    logs = json.load(f)

for ep in logs:
    if ep["reward"] > 0 and ep.get("disqualification_reason") is None:
        prompt = f"<|system|>\nYou are an expert database administrator.\n<|user|>\nOptimize this SQL for the `{ep['database']}` schema:\n{ep['baseline_sql']}\n<|assistant|>\n"
        
        chosen_resp = f"<reasoning>\n{ep['reasoning_trace']}\n</reasoning>\n{ep['optimized_sql']}"
        rejected_resp = f"<reasoning>\nSequential scan is fine.\n</reasoning>\n{ep['baseline_sql']}"
        
        pairs["prompt"].append(prompt)
        pairs["chosen"].append(chosen_resp + tokenizer.eos_token)
        pairs["rejected"].append(rejected_resp + tokenizer.eos_token)

print(f"Extracted {len(pairs['prompt'])} valid preference pairs.")
dataset = Dataset.from_dict(pairs)

Extracted 16 valid preference pairs.

## 3. Exp A: Reward Model Training (TRL)
Experimenting with a standalone reward model to evaluate penalization of `LIMIT 0` hacks prior to PPO implementation.

In [ ]:
from trl import RewardTrainer, RewardConfig
from transformers import AutoModelForSequenceClassification

# rm_model = AutoModelForSequenceClassification.from_pretrained(
#     model_id, num_labels=1, quantization_config=bnb_config, device_map="auto"
# )

# rm_config = RewardConfig(
#     output_dir="./rm_checkpoints",
#     per_device_train_batch_size=2,
#     learning_rate=1e-5,
# )

# rm_trainer = RewardTrainer(
#     model=rm_model,
#     args=rm_config,
#     train_dataset=dataset,
#     tokenizer=tokenizer,
# )

# rm_trainer.train()

# // --- NOTE ---
# RM convergence is insufficient. Margin between chosen/rejected fluctuating around 0.1.
# High noise floor will destabilize PPO.
# Abandoning standalone RM approach. Pivoting directly to DPO.

## 4. Exp B: Direct Preference Optimization (DPO)
Optimizing policy directly on preference data using DPO.
Initializing with `beta=0.1` for regularization.

In [ ]:
from trl import DPOTrainer
from transformers import TrainingArguments

dpo_args = TrainingArguments(
    output_dir="./dpo_tinyllama_sql",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=3,
    optim="paged_adamw_8bit",
    logging_steps=5,
    remove_unused_columns=False
)

dpo_trainer = DPOTrainer(
    model,
    ref_model=None,
    args=dpo_args,
    beta=0.1,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_prompt_length=256,
    max_length=512
)

print("Starting DPO training...")
# dpo_trainer.train()

Starting DPO training...

Loss reduction confirmed.
Manual generation validation required to check for schema overfitting.

## 5. Export and Serving
Merging LoRA weights into base model for GGUF conversion and local deployment via Ollama.

In [ ]:
# Merging adapters
# merged = dpo_trainer.model.merge_and_unload()
# merged.save_pretrained("./tinyllama_sql_merged")
# tokenizer.save_pretrained("./tinyllama_sql_merged")

# Converting to GGUF
# !python3 llama.cpp/convert.py ./tinyllama_sql_merged --outfile tinyllama_sql_dpo.gguf --outtype q4_0

# Generating Modelfile
modelfile = """
FROM ./tinyllama_sql_dpo.gguf
SYSTEM "You are an expert SQL performance tuning engine."
"""
with open("Modelfile", "w") as f:
    f.write(modelfile)

# TODO: execute `ollama create local-sql-dpo -f Modelfile`